# Full-Study Output Analytics (v2)

Computes **refusal rates**, **claim density**, and **hallucination correlations** on the
full-study dataset, distinguishing two distinct corpus measurements:

| Corpus               | What it measures                                            | Source file                  |
|----------------------|-------------------------------------------------------------|------------------------------|
| **Final-answer**     | Words in the final synthesized reports the LLM panel judged | `all_triplets_cache.csv`     |
| **Generation**       | Every word produced across ALL stages of multi-stage prompting | `generation_corpus_full.json` (precomputed) |

Final-answer corpus = ~17.3M words (9,680 reports). Generation corpus = ~64.1M words (9,743 stage runs).

**Outputs (saved to `C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\`):**
- `word_count_final_answers.json` — final-answer-corpus word stats
- `refusal_rate_full.csv` — refusal rate per (model, technique)
- `claim_density_full.csv` — claim density (from generation corpus) × hallucination rate
- `correlations_full.json` — Spearman correlations between metrics
- `metrics_full.csv` — combined per-cell metrics
- `summary_full.txt` — paper-ready summary

All sections can be run independently after Cell 2.

In [1]:
import os, json, re
import numpy as np
import pandas as pd
from scipy import stats

OUTPUT_DIR    = r'C:\Opeyemi\PROMPTS\EVALUATION'
TRIPLETS_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
PANEL_RAW     = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')

WC_DIR        = r'C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT'
os.makedirs(WC_DIR, exist_ok=True)

# Inputs (precomputed elsewhere)
GENERATION_CORPUS_JSON = os.path.join(WC_DIR, 'generation_corpus_full.json')

# Outputs from this notebook
WORD_COUNT_OUT     = os.path.join(WC_DIR, 'word_count_final_answers.json')
REFUSAL_OUT        = os.path.join(WC_DIR, 'refusal_rate_full.csv')
CLAIM_DENSITY_OUT  = os.path.join(WC_DIR, 'claim_density_full.csv')
CORRELATIONS_OUT   = os.path.join(WC_DIR, 'correlations_full.json')
SUMMARY_OUT        = os.path.join(WC_DIR, 'summary_full.txt')
METRICS_OUT        = os.path.join(WC_DIR, 'metrics_full.csv')

HCOLS = ['H1','H2','H3','H4','H5','H6']

# Load triplets (final answers)
df = pd.read_csv(TRIPLETS_PATH)
df['model_output'] = df['model_output'].fillna('').astype(str)

# Load generation corpus (precomputed)
if not os.path.exists(GENERATION_CORPUS_JSON):
    print(f'WARNING: {GENERATION_CORPUS_JSON} not found.')
    print('  Run the generation_corpus_counter notebook first to produce it.')
    print('  Sections that need generation-corpus data will fail until you do.')
    gen_corpus = None
else:
    with open(GENERATION_CORPUS_JSON, 'r') as f:
        gen_corpus = json.load(f)
    print(f'Generation corpus loaded: {gen_corpus["grand_totals"]["words"]:,} words across '
          f'{gen_corpus["grand_totals"]["experiments"]:,} experiments')

print(f'\nTriplets (final answers): {len(df):,} rows')
print(f'  Models:     {sorted(df["model"].unique().tolist())}')
print(f'  Techniques: {sorted(df["technique"].unique().tolist())}')
print(f'\nPer (model, technique) counts:')
print(df.groupby(["model","technique"]).size().unstack(fill_value=0))


Generation corpus loaded: 64,073,675 words across 9,743 experiments

Triplets (final answers): 9,680 rows
  Models:     ['Claude', 'GPT', 'Gemini']
  Techniques: ['Least-to-Most', 'ReAct', 'Sequential', 'Zero-Shot']

Per (model, technique) counts:
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude               807    807         804        807
GPT                  807    807         807        807
Gemini               806    807         807        807


---
## 1. Final-answer corpus (words in the reports the panel judged)

In [2]:
SENT_RE  = re.compile(r'[.!?]+(?:\s|$)')
CLAIM_RE = re.compile(r'(?:[.!?;]|--|—|\n\s*[-*•])')

def count_words(text): return len(text.split()) if text else 0
def count_sentences(text):
    return sum(1 for p in SENT_RE.split(text) if p.strip()) if text else 0
def count_claims(text):
    if not text: return 0
    parts = [p.strip() for p in CLAIM_RE.split(text) if p.strip()]
    return sum(1 for p in parts if len(p.split()) >= 3)

print('Counting words / sentences / claims for all final-answer outputs...')
df['word_count']  = df['model_output'].apply(count_words)
df['sent_count']  = df['model_output'].apply(count_sentences)
df['claim_count'] = df['model_output'].apply(count_claims)

# Per-model totals
model_totals = df.groupby('model').agg(
    experiments=('model','size'),
    words=('word_count','sum'),
    sentences=('sent_count','sum'),
    claims=('claim_count','sum'),
).to_dict('index')

# Per (model, technique)
tech_details = {}
for model in df['model'].unique():
    tech_details[model] = {}
    for tech in df[df['model']==model]['technique'].unique():
        sub = df[(df['model']==model) & (df['technique']==tech)]
        tech_details[model][tech] = {
            'experiments': int(len(sub)),
            'words': int(sub['word_count'].sum()),
            'sentences': int(sub['sent_count'].sum()),
            'claims': int(sub['claim_count'].sum()),
            'mean_words': round(float(sub['word_count'].mean()), 1),
        }

grand = {
    'experiments': int(len(df)),
    'words': int(df['word_count'].sum()),
    'sentences': int(df['sent_count'].sum()),
    'claims': int(df['claim_count'].sum()),
}

final_answer_data = {
    'source': TRIPLETS_PATH,
    'description': 'Final-answer corpus: words in the synthesized final reports judged by the LLM panel.',
    'n_rows': int(len(df)),
    'model_results': {m: {k: int(v) for k, v in d.items()} for m, d in model_totals.items()},
    'grand_totals': grand,
    'technique_details': tech_details,
}

with open(WORD_COUNT_OUT, 'w') as f:
    json.dump(final_answer_data, f, indent=2)
print(f'\nSaved: {WORD_COUNT_OUT}')

print('\n=== FINAL-ANSWER CORPUS SUMMARY ===')
print(f'Grand total: {grand["words"]:,} words across {grand["experiments"]:,} reports')
print('\nPer-model:')
for m, d in model_totals.items():
    print(f'  {m:<8} reports={d["experiments"]:>5,}  words={d["words"]:>12,}  '
          f'claims={d["claims"]:>10,}')
print('\nPer (model, technique):')
print(f'{"Model":<10}{"Technique":<16}{"n":>5}{"Words":>12}{"MeanW":>8}{"Claims":>10}')
print('-'*60)
for model, techs in tech_details.items():
    for tech, d in techs.items():
        print(f'{model:<10}{tech:<16}{d["experiments"]:>5,}{d["words"]:>12,}'
              f'{d["mean_words"]:>8.1f}{d["claims"]:>10,}')


Counting words / sentences / claims for all final-answer outputs...

Saved: C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\word_count_final_answers.json

=== FINAL-ANSWER CORPUS SUMMARY ===
Grand total: 17,342,872 words across 9,680 reports

Per-model:
  Claude   reports=3,225  words=   3,079,500  claims=   253,802
  GPT      reports=3,228  words=   4,580,663  claims=   337,489
  Gemini   reports=3,227  words=   9,682,709  claims=   593,363

Per (model, technique):
Model     Technique           n       Words   MeanW    Claims
------------------------------------------------------------
Claude    Zero-Shot         807     543,425   673.4    39,896
Claude    Sequential        804     934,942  1162.9    90,220
Claude    Least-to-Most     807     687,569   852.0    53,403
Claude    ReAct             807     913,564  1132.0    70,283
GPT       Zero-Shot         807     153,483   190.2     7,928
GPT       Sequential        807     281,479   348.8    17,926
GPT       Least-to-Most     807     366,184  

---
## 2. Refusal rate by (model, technique)

In [3]:
refusal_phrases = [
    'i cannot', "i can't", "i'm unable", 'i am unable', "i'm not able",
    'cannot analyze', 'cannot process', 'not appropriate', 'i must decline',
    'i apologize', 'unable to provide', 'cannot provide', 'i refuse',
    "i'm sorry, but", 'against my guidelines', 'potentially harmful',
    'sensitive content', 'cannot assist', 'not able to assist',
    'i cannot fulfill', 'inappropriate', "i won't", 'i will not',
]

def is_refusal(text):
    if not text: return False
    t = text.lower()
    return any(p in t for p in refusal_phrases)

df['is_refusal'] = df['model_output'].apply(is_refusal)

print('=== Refusal rate by (model, technique) ===')
print(f'{"Model":<10}{"Technique":<16}{"n":>5}{"refusals":>10}{"rate":>8}{"mean_words":>12}')
print('-'*61)
ref_rows = []
for (m, t), grp in df.groupby(['model','technique']):
    n = len(grp)
    n_ref = int(grp['is_refusal'].sum())
    rate = n_ref / n * 100 if n else 0
    mean_w = grp['word_count'].mean()
    flag = ' <<<' if rate > 30 else ''
    print(f'{m:<10}{t:<16}{n:>5,}{n_ref:>10}{rate:>7.1f}%{mean_w:>11.0f}{flag}')
    ref_rows.append({
        'model': m, 'technique': t,
        'n': n, 'refusals': n_ref,
        'refusal_rate_pct': round(rate, 2),
        'mean_words_final': round(float(mean_w), 1),
    })

ref_df = pd.DataFrame(ref_rows)
ref_df.to_csv(REFUSAL_OUT, index=False)
print(f'\nSaved: {REFUSAL_OUT}')

print('\n=== Overall refusal rate per model (final answers) ===')
for m in df['model'].unique():
    sub = df[df['model']==m]
    rate = sub['is_refusal'].mean() * 100
    print(f'  {m:<10} {sub["is_refusal"].sum()}/{len(sub)} ({rate:.1f}%)')

# Sample one refusal per model
print('\n=== Sample refusals (first 250 chars) ===')
for m in df['model'].unique():
    sub = df[(df['model']==m) & (df['is_refusal'])]
    if len(sub) > 0:
        s = sub.iloc[0]
        print(f'\n  {m} / {s["technique"]} / {s["video"]}:')
        print(f'    "{s["model_output"][:250]}..."')


=== Refusal rate by (model, technique) ===
Model     Technique           n  refusals    rate  mean_words
-------------------------------------------------------------
Claude    Least-to-Most     807        15    1.9%        852
Claude    ReAct             807         4    0.5%       1132
Claude    Sequential        804        31    3.9%       1163
Claude    Zero-Shot         807         3    0.4%        673
GPT       Least-to-Most     807         6    0.7%        454
GPT       ReAct             807       118   14.6%       4683
GPT       Sequential        807         8    1.0%        349
GPT       Zero-Shot         807         2    0.2%        190
Gemini    Least-to-Most     806        28    3.5%       2858
Gemini    ReAct             807       175   21.7%       5737
Gemini    Sequential        807         2    0.2%        462
Gemini    Zero-Shot         807         6    0.7%       2944

Saved: C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\refusal_rate_full.csv

=== Overall refusal rate per mode

---
## 3. Claim density (from generation corpus) vs hallucination rate

Claim density is computed against the **generation corpus** (all stages summed), matching
the methodology used in the pilot study. This captures how dense the model is with factual
claims throughout its reasoning, not just in the final answer.

In [4]:
if gen_corpus is None:
    raise RuntimeError(
        'generation_corpus_full.json is missing. '
        'Run the generation_corpus_counter_v2 notebook first.'
    )

# Load panel labels and aggregate hallucination rates per (model, technique)
labels = pd.read_csv(PANEL_RAW)
print(f'Panel raw labels: {len(labels):,} rows')

for h in HCOLS:
    labels[h] = pd.to_numeric(labels[h], errors='coerce')

clean = labels[(labels[HCOLS] >= 0).all(axis=1)].copy()
print(f'Clean panel rows (no -1): {len(clean):,}')

# Majority vote per row across judges
def majority(col):
    return col.mode().iloc[0] if not col.mode().empty else 0

mv = clean.groupby(['row_idx','model','technique'])[HCOLS].agg(majority).reset_index()
print(f'Triplets with majority labels: {len(mv):,}')

# Per (model, technique) hallucination metrics
agg_rows = []
for (m, t), grp in mv.groupby(['model','technique']):
    row = {'model': m, 'technique': t, 'n_triplets': len(grp)}
    for h in HCOLS:
        row[f'{h}_rate'] = round(grp[h].mean(), 4)
    row['hall_total'] = int(grp[HCOLS].sum().sum())
    row['hall_per_row'] = round(grp[HCOLS].sum().sum() / len(grp), 4)
    row['any_rate'] = round((grp[HCOLS].sum(axis=1) > 0).mean(), 4)
    agg_rows.append(row)
hall_df = pd.DataFrame(agg_rows)

# Build generation-corpus per-cell frame
gen_rows = []
for model, techs in gen_corpus['technique_details'].items():
    for tech, d in techs.items():
        gen_rows.append({
            'model': model, 'technique': tech,
            'gen_words': d['words'],
            'gen_claims': d['claims'],
            'gen_sentences': d['sentences'],
            'gen_mean_words': d['mean_words'],
            'gen_experiments': d['experiments'],
        })
gen_df = pd.DataFrame(gen_rows)
gen_df['claim_density'] = gen_df['gen_claims'] / gen_df['gen_words']

# Merge
cd_df = gen_df.merge(hall_df, on=['model','technique'], how='outer')
cd_df = cd_df.sort_values(['model','technique']).reset_index(drop=True)
cd_df.to_csv(CLAIM_DENSITY_OUT, index=False)
print(f'\nSaved: {CLAIM_DENSITY_OUT}')

print('\n=== CLAIM DENSITY (from generation corpus) vs HALLUCINATION RATE ===')
print(f'{"Model":<10}{"Technique":<16}{"GenWords":>12}{"Claims":>10}{"Density":>9}'
      f'{"Hall/row":>10}{"AnyRate":>9}')
print('-'*76)
for _, r in cd_df.iterrows():
    print(f'{r["model"]:<10}{r["technique"]:<16}{int(r["gen_words"]):>12,}'
          f'{int(r["gen_claims"]):>10,}{r["claim_density"]:>9.4f}'
          f'{r["hall_per_row"]:>10.3f}{r["any_rate"]:>9.3f}')


Panel raw labels: 19,360 rows
Clean panel rows (no -1): 19,358
Triplets with majority labels: 9,680

Saved: C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\claim_density_full.csv

=== CLAIM DENSITY (from generation corpus) vs HALLUCINATION RATE ===
Model     Technique           GenWords    Claims  Density  Hall/row  AnyRate
----------------------------------------------------------------------------
Claude    Least-to-Most     26,606,855 2,261,923   0.0850     2.699    0.963
Claude    ReAct              5,962,808   526,456   0.0883     3.156    1.000
Claude    Sequential         2,668,458   225,578   0.0845     2.791    0.980
Claude    Zero-Shot          2,360,882   191,609   0.0812     3.172    1.000
GPT       Least-to-Most      1,963,452   140,869   0.0717     2.125    0.885
GPT       ReAct              3,874,123   308,618   0.0797     1.208    0.897
GPT       Sequential         2,397,043   146,474   0.0611     1.693    0.731
GPT       Zero-Shot            706,652    46,328   0.0656     1.285  

---
## 4. Correlations between metrics

In [5]:
# Combine claim density, hallucination, refusal, mean-words into one frame
metrics = cd_df.merge(ref_df[['model','technique','refusal_rate_pct','mean_words_final']],
                       on=['model','technique'], how='left')
metrics['refusal_rate'] = metrics['refusal_rate_pct'] / 100.0

correlations = {}
print('=== Spearman correlations across (model, technique) cells (n=12) ===\n')

pairs = [
    ('claim_density',     'hall_per_row',  'Claim density vs Halls per row'),
    ('claim_density',     'any_rate',      'Claim density vs ANY-hall rate'),
    ('gen_mean_words',    'hall_per_row',  'Mean words per generation vs Halls per row'),
    ('gen_mean_words',    'any_rate',      'Mean words per generation vs ANY-hall rate'),
    ('mean_words_final',  'hall_per_row',  'Mean words (final) vs Halls per row'),
    ('mean_words_final',  'any_rate',      'Mean words (final) vs ANY-hall rate'),
    ('refusal_rate',      'hall_per_row',  'Refusal rate vs Halls per row'),
    ('refusal_rate',      'any_rate',      'Refusal rate vs ANY-hall rate'),
    ('gen_words',         'gen_claims',    'Total gen words vs Total gen claims'),
]
for x, y, label in pairs:
    valid = metrics[[x, y]].dropna()
    if len(valid) < 3:
        print(f'  {label}: too few rows')
        continue
    rho, p = stats.spearmanr(valid[x], valid[y])
    correlations[label] = {'rho': round(float(rho), 4), 'p': round(float(p), 4),
                           'n': int(len(valid))}
    print(f'  {label:<46} rho = {rho:+.3f}  p = {p:.3f}  (n={len(valid)})')

# Per-H correlations with claim density
print('\n=== Claim density vs each H-type rate ===')
for h in HCOLS:
    col = f'{h}_rate'
    valid = metrics[['claim_density', col]].dropna()
    if len(valid) < 3:
        continue
    rho, p = stats.spearmanr(valid['claim_density'], valid[col])
    correlations[f'Claim density vs {h}'] = {'rho': round(float(rho), 4),
                                               'p': round(float(p), 4),
                                               'n': int(len(valid))}
    print(f'  {h}: rho = {rho:+.3f}  p = {p:.3f}')

with open(CORRELATIONS_OUT, 'w') as f:
    json.dump(correlations, f, indent=2)
print(f'\nSaved: {CORRELATIONS_OUT}')

metrics.to_csv(METRICS_OUT, index=False)
print(f'Saved: {METRICS_OUT}')


=== Spearman correlations across (model, technique) cells (n=12) ===

  Claim density vs Halls per row                 rho = +0.559  p = 0.059  (n=12)
  Claim density vs ANY-hall rate                 rho = +0.760  p = 0.004  (n=12)
  Mean words per generation vs Halls per row     rho = +0.378  p = 0.226  (n=12)
  Mean words per generation vs ANY-hall rate     rho = +0.322  p = 0.307  (n=12)
  Mean words (final) vs Halls per row            rho = +0.140  p = 0.665  (n=12)
  Mean words (final) vs ANY-hall rate            rho = +0.210  p = 0.512  (n=12)
  Refusal rate vs Halls per row                  rho = +0.011  p = 0.974  (n=12)
  Refusal rate vs ANY-hall rate                  rho = +0.039  p = 0.905  (n=12)
  Total gen words vs Total gen claims            rho = +0.958  p = 0.000  (n=12)

=== Claim density vs each H-type rate ===
  H1: rho = +0.424  p = 0.170
  H2: rho = +0.462  p = 0.131
  H3: rho = +0.301  p = 0.342
  H4: rho = -0.510  p = 0.090
  H5: rho = +0.382  p = 0.221
  H6: rh

---
## 5. Paper-ready summary

In [6]:
lines = []
lines.append('=' * 72)
lines.append('FULL-STUDY OUTPUT ANALYTICS — SUMMARY')
lines.append('=' * 72)
lines.append('')
lines.append(f'Final-answer corpus (reports judged by panel):')
lines.append(f'  Reports:    {grand["experiments"]:>15,}')
lines.append(f'  Words:      {grand["words"]:>15,}')
lines.append(f'  Claims:     {grand["claims"]:>15,}')
lines.append('')
gen_g = gen_corpus["grand_totals"]
lines.append(f'Generation corpus (all stages summed):')
lines.append(f'  Experiments:{gen_g["experiments"]:>15,}')
lines.append(f'  Words:      {gen_g["words"]:>15,}')
lines.append(f'  Claims:     {gen_g["claims"]:>15,}')
lines.append('')
lines.append('-- REFUSAL RATES (final answers) --')
for m in sorted(df['model'].unique()):
    sub = df[df['model']==m]
    rate = sub['is_refusal'].mean() * 100
    lines.append(f'  {m:<10} {sub["is_refusal"].sum():>4}/{len(sub):>4} ({rate:>5.1f}%)')
lines.append('')
lines.append('-- CLAIM DENSITY (gen claims / gen words) --')
for _, r in cd_df.sort_values(['model','technique']).iterrows():
    lines.append(f'  {r["model"]:<10}{r["technique"]:<16} {r["claim_density"]:.4f}')
lines.append('')
lines.append('-- KEY CORRELATIONS --')
for label, vals in correlations.items():
    lines.append(f'  {label:<48} rho={vals["rho"]:+.3f}  p={vals["p"]:.3f}')

summary = '\n'.join(lines)
with open(SUMMARY_OUT, 'w') as f:
    f.write(summary)
print(summary)
print(f'\nSaved: {SUMMARY_OUT}')


FULL-STUDY OUTPUT ANALYTICS — SUMMARY

Final-answer corpus (reports judged by panel):
  Reports:              9,680
  Words:           17,342,872
  Claims:           1,184,654

Generation corpus (all stages summed):
  Experiments:          9,743
  Words:           64,073,675
  Claims:           4,884,153

-- REFUSAL RATES (final answers) --
  Claude       53/3225 (  1.6%)
  GPT         134/3228 (  4.2%)
  Gemini      211/3227 (  6.5%)

-- CLAIM DENSITY (gen claims / gen words) --
  Claude    Least-to-Most    0.0850
  Claude    ReAct            0.0883
  Claude    Sequential       0.0845
  Claude    Zero-Shot        0.0812
  GPT       Least-to-Most    0.0717
  GPT       ReAct            0.0797
  GPT       Sequential       0.0611
  GPT       Zero-Shot        0.0656
  Gemini    Least-to-Most    0.0590
  Gemini    ReAct            0.0626
  Gemini    Sequential       0.0546
  Gemini    Zero-Shot        0.0567

-- KEY CORRELATIONS --
  Claim density vs Halls per row                   rho=+0.5